<a href="https://colab.research.google.com/github/prodigal94/food-safe-bots/blob/main/Week_3_Linear_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql import SparkSession
from pyspark.ml.regression import LinearRegression

In [2]:
# Initialize Spark with optimizations for large data
spark = SparkSession.builder \
    .appName("BDA_Project") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .getOrCreate()

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
df = spark.read.parquet("/content/drive/MyDrive/Parquet/02_AgriTrade_ValueOnly.parquet")

In [5]:
train, test = df.randomSplit([.8,.2], seed = 6942067)
print(f"Train Data: {train.count()}, Test Data: {test.count()}")

Train Data: 3451681, Test Data: 863720


In [14]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline

country_indexer = StringIndexer(inputCol="Area", outputCol="country_index")
item_indexer = StringIndexer(inputCol="Item", outputCol="item_index")

encoder = OneHotEncoder(
    inputCols=["country_index", "item_index"],
    outputCols=["country_vec", "item_vec"]
)

assembler = VectorAssembler(
    inputCols=["Year", "country_vec", "item_vec"],
    outputCol="features"
)

lrModel = LinearRegression(featuresCol="features", labelCol="Value")
pipeline = Pipeline(stages=[
    country_indexer,
    item_indexer,
    encoder,
    assembler,
    lrModel
])

model = pipeline.fit(train)

predictions = model.transform(test)
predictions.select("Area", "Item", "Year", "Value", "Prediction").orderBy("Value", ascending=False).show()

+--------------------+--------------------+----+-------------+--------------------+
|                Area|                Item|Year|        Value|          Prediction|
+--------------------+--------------------+----+-------------+--------------------+
|               China|Total Merchandise...|2022|4.682902787E9| 5.555149420000504E7|
|               China|Total Merchandise...|2023|4.387886378E9|  5.55812047857708E7|
|China (excluding ...|Total Merchandise...|2023|4.387886378E9| 5.634248874406712E7|
|               China|Total Merchandise...|2022|3.836878216E9| 5.555149420000504E7|
|China (excluding ...|Total Merchandise...|2022|3.836878216E9| 5.631277815830137E7|
|China (excluding ...|Total Merchandise...|2021|3.801207338E9|5.6283067572535604E7|
|China (excluding ...|Total Merchandise...|2015|   3.070747E9|5.6104804057941034E7|
|China (excluding ...|Total Merchandise...|2018|3.059858613E9| 5.619393581523831E7|
|               China|Total Merchandise...|2014|   2.853107E9| 5.53138095138

In [15]:
### Evaluation
lr_model = model.stages[-1]
training_summary = lr_model.summary

print(f"RMSE: {training_summary.rootMeanSquaredError}")
print(f"R2: {training_summary.r2}")

RMSE: 17619546.36461125
R2: 0.04692553284938461
